In [82]:
!pip install pyspark

In [83]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [84]:
from google.colab import drive
drive.mount('/content/drive')

In [85]:
spark = SparkSession.builder.getOrCreate()

In [86]:
df_video = spark.read.parquet('/content/videos-preparados.snappy.parquet')

In [87]:
df_video.show(10)

+--------------------+-----------+------------+---------+------+--------+--------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+
|               Title|   Video ID|Published At|  Keyword| Likes|Comments|   Views|Interaction|Year|Month|Keyword Index|        Features PCA|     Features Normal|            Features|
+--------------------+-----------+------------+---------+------+--------+--------+-----------+----+-----+-------------+--------------------+--------------------+--------------------+
|ASMR MUKBANG DOUB...|--ZI0dSbbNU|  2020-04-18|  mukbang|378858|   18860|17975269|   18372987|2020|    4|         30.0|[0.6985786560867407]|[0.02303716158264...|[378858.0,1.79752...|
|Deadly car bomb d...|--hxd1CrOqg|  2022-08-22|     news|  6379|    4853|  808787|     820019|2022|    8|         37.0|[0.8936407990235931]|[3.87946679100418...|[6379.0,808787.0,...|
|How Biden&#39;s s...|--ixiTypG8g|  2022-08-24|     news|  1029|    2347|   97434|   

In [88]:
keyword_count = df_video.groupBy('Keyword').count() # calculando a quantidade de registros para cada valor único da coluna "Keyword"
keyword_count.show()

+----------------+-----+
|         Keyword|count|
+----------------+-----+
|computer science|   48|
|            lofi|   40|
|         finance|   39|
|             cnn|   50|
|           apple|   42|
|            news|   39|
|         mukbang|   45|
|       education|   24|
|       interview|   50|
|          crypto|   50|
|   mathchemistry|   15|
|            food|   48|
|    data science|   50|
|        trolling|   50|
|        tutorial|   50|
|      literature|   46|
|             sat|   49|
|         history|   49|
|           cubes|   49|
|           music|   46|
+----------------+-----+
only showing top 20 rows


In [89]:
keyword_interaction_df = df_video.groupBy('Keyword').avg('Interaction') # calculando a média da coluna "Interaction" para cada valor único da coluna 'Keyword'
keyword_interaction_df.show()

+----------------+--------------------+
|         Keyword|    avg(Interaction)|
+----------------+--------------------+
|computer science|  1226793.0208333333|
|            lofi|         4167085.875|
|         finance|   708542.9487179487|
|             cnn|           570650.86|
|           apple|1.0873628214285715E7|
|            news|  251688.71794871794|
|         mukbang|1.1053630377777778E7|
|       education|         2750838.625|
|       interview|          3044867.04|
|          crypto|            413676.2|
|   mathchemistry|  3427342.7333333334|
|            food|   5352944.104166667|
|    data science|           562465.28|
|        trolling|          1484584.88|
|        tutorial|           6936688.3|
|      literature|            881726.5|
|             sat|           1098927.0|
|         history| 1.565269257142857E7|
|           cubes|1.5043961224489795E7|
|           music|2.9691370304347824E7|
+----------------+--------------------+
only showing top 20 rows


In [90]:
rank_interactions_df = df_video.groupBy('Keyword').agg(

    max('Interaction').alias('Rank Interactions') # calculando o valor máximo da coluna "Interaction" e nomeando a coluna para 'Rank Interactions'

).orderBy(col('Rank Interactions').desc()).show() # agrupando pela nova coluna em ordem decrescente

+--------+-----------------+
| Keyword|Rank Interactions|
+--------+-----------------+
| animals|       1593623628|
|   music|        922551152|
|     bed|        532691631|
| history|        440187490|
|   apple|        429916936|
| mrbeast|        300397699|
|  google|        239385460|
|business|        210025196|
|   cubes|        170925917|
|  sports|        106924567|
| mukbang|         87433858|
|    lofi|         86445177|
|tutorial|         69616442|
|  movies|         65253870|
|  marvel|         56247330|
|  how-to|         53053975|
|    food|         48754479|
| physics|         43463298|
|    asmr|         34411125|
|nintendo|         32268486|
+--------+-----------------+
only showing top 20 rows


In [91]:
views_keyword_df = df_video.groupBy('Keyword').agg(
    avg('Views').alias('Views Médio'), # calculando a média da coluna 'Views' para cada valor único da coluna 'Keyword' e renomeando a coluna
    var_samp('Views').alias('Views Variancia') # calculando a variância da coluna 'Views' para cada valor único da coluna 'Keyword' e renomeando a coluna
).show()

+----------------+--------------------+--------------------+
|         Keyword|         Views Médio|     Views Variancia|
+----------------+--------------------+--------------------+
|computer science|  1191958.7083333333| 2.81219868165102E12|
|            lofi|           4089363.0|1.846209641476677...|
|         finance|   694223.4358974359|3.304483175097042...|
|             cnn|           554240.38|1.563423618468118...|
|           apple|1.0746930452380951E7|4.299927977442589E15|
|            news|   247492.1794871795|1.067512576672564...|
|         mukbang|1.0904772355555555E7|5.586073238973179...|
|       education|  2684432.8333333335|1.833572249339214...|
|       interview|          2966111.86|1.819220996034335E13|
|          crypto|           404608.22|3.513691634369074E12|
|   mathchemistry|  3328125.2666666666|2.491467065256849...|
|            food|          5252406.25|7.326374128154842E13|
|    data science|           544771.98|5.479336525349994...|
|        trolling|      

In [92]:
views_keyword_semdecimal_df = df_video.groupBy('Keyword').agg(
    format_number(avg('Views'),0).alias('Views Média'), # Calculando a média de 'Views' para cada valor único da coluna 'Keyword', sem casas decimais
    format_number(min('Views'),0).alias('Views Mínimo'), # Calculando o mínimo de 'Views' para cada valor único da coluna 'Keyword', sem casas decimais
    format_number(max('Views'),0).alias('Views Máximo') # Calculando o máximo de 'Views' para cada valor único da coluna 'Keyword', sem casas decimais
).show()

+----------------+-----------+------------+------------+
|         Keyword|Views Média|Views Mínimo|Views Máximo|
+----------------+-----------+------------+------------+
|computer science|  1,191,959|      16,115|   7,004,107|
|            lofi|  4,089,363|       6,817|  84,747,957|
|         finance|    694,223|       1,195|   9,450,554|
|             cnn|    554,240|      51,269|   1,889,320|
|           apple| 10,746,930|       1,954| 425,478,119|
|            news|    247,492|      10,642|   1,465,011|
|         mukbang| 10,904,772|       3,618|  86,169,225|
|       education|  2,684,433|       6,611|  17,103,736|
|       interview|  2,966,112|       2,587|  22,529,756|
|          crypto|    404,608|       1,599|  11,805,668|
|   mathchemistry|  3,328,125|          25|  18,496,859|
|            food|  5,252,406|      47,430|  48,018,833|
|    data science|    544,772|         911|   3,069,097|
|        trolling|  1,420,141|       5,388|  14,286,302|
|        tutorial|  6,761,032| 

In [93]:
data_published_df = df_video.groupBy('Keyword').agg(
    first('Published At').alias('Primeira data'), # mostrando primeiro 'Published At' para cada valor único da coluna 'Keyword' e modificando o nome da coluna
    last('Published At').alias('Última data') # mostrando último 'Published At' para cada valor único da coluna 'Keyword' e modificando o nome da coluna
).show()

+----------------+-------------+-----------+
|         Keyword|Primeira data|Última data|
+----------------+-------------+-----------+
|computer science|   2022-02-08| 2020-09-08|
|            lofi|   2022-06-07| 2020-07-19|
|         finance|   2020-09-23| 2017-12-31|
|             cnn|   2022-08-17| 2022-08-13|
|           apple|   2022-08-22| 2022-08-02|
|            news|   2022-08-22| 2022-08-23|
|         mukbang|   2020-04-18| 2022-08-24|
|       education|   2015-02-06| 2010-10-14|
|       interview|   2021-08-03| 2018-10-05|
|          crypto|   2022-08-23| 2022-08-22|
|   mathchemistry|   2020-08-11| 2019-10-04|
|            food|   2022-07-17| 2022-08-20|
|    data science|   2019-08-18| 2021-08-06|
|        trolling|   2022-08-23| 2022-07-20|
|        tutorial|   2018-06-01| 2022-08-17|
|      literature|   2016-08-05| 2016-01-15|
|             sat|   2022-03-11| 2016-06-16|
|         history|   2022-04-13| 2016-01-26|
|           cubes|   2022-04-20| 2021-12-03|
|         

In [94]:
total_title_count = df_video.count()

# Contando titles únicos
unicos_title_count = df_video.agg(countDistinct('Title')).collect()[0][0]

# Calculando a diferença
diferenca = total_title_count - unicos_title_count

print(f'Total de registros (titles): {total_title_count}')
print(f'Títulos únicos: {unicos_title_count}')
print(f'Diferença (duplicados): {diferenca}')

Total de registros (titles): 1869
Títulos únicos: 1854
Diferença (duplicados): 15


In [99]:
df_video.withColumn('anos_ordenados', year('Published At')
).groupBy('anos_ordenados' # agrupando a coluna de 'anos_ordenados'
).count().orderBy(asc('anos_ordenados') # mostrando a quantidade de registros ordenados por ano em ordem ascendente
).show()

+--------------+-----+
|anos_ordenados|count|
+--------------+-----+
|          2007|    2|
|          2008|    1|
|          2009|    9|
|          2010|    6|
|          2011|    4|
|          2012|   12|
|          2013|    6|
|          2014|   10|
|          2015|   15|
|          2016|   34|
|          2017|   47|
|          2018|   57|
|          2019|   86|
|          2020|  158|
|          2021|  229|
|          2022| 1193|
+--------------+-----+



In [96]:
df_video.withColumn('anos_ordenados', year('Published At') # criando uma nova coluna chamada 'anos_ordenados' que extrai o ano da coluna 'Published At'
).withColumn('meses_ordenados', month('Published At') # criando uma nova coluna chamada 'meses_ordenados' que extrai os meses da coluna 'Published At'
).groupBy('anos_ordenados', 'meses_ordenados' # agrupa os dados por ano e mês
).count().orderBy(asc('anos_ordenados'),'meses_ordenados' # contando quantos vídeos existem em cada grupo e colocando em ordem cronológica
).show()


+--------------+---------------+-----+
|anos_ordenados|meses_ordenados|count|
+--------------+---------------+-----+
|          2007|              7|    1|
|          2007|             12|    1|
|          2008|              7|    1|
|          2009|              2|    2|
|          2009|              6|    2|
|          2009|              7|    1|
|          2009|              8|    1|
|          2009|             10|    1|
|          2009|             12|    2|
|          2010|              3|    1|
|          2010|              5|    2|
|          2010|              6|    1|
|          2010|              9|    1|
|          2010|             10|    1|
|          2011|              2|    1|
|          2011|              5|    1|
|          2011|              9|    1|
|          2011|             10|    1|
|          2012|              1|    1|
|          2012|              2|    3|
+--------------+---------------+-----+
only showing top 20 rows


In [97]:
df_video = df_video.withColumn("Ano", year(col("Published At"))) # criando uma coluna Ano extraindo o ano da coluna Published At

acumulativa = Window.partitionBy('Keyword').orderBy('Ano' # separando os dados por palavra-chave e ordenando dentro de cada grupo por ano
).rowsBetween(Window.unboundedPreceding, Window.currentRow)
df_resultado = df_video.withColumn(
    'Media_Likes_Acumulativa', # criando a coluna 'Media_Likes_Acumulativa'
    avg('Likes').over(acumulativa)) # calculando a média dos likes dentro dessa janela

df_resultado.select(
    'Keyword',
    'Ano',
    'Likes',
    'Media_Likes_Acumulativa' # selecionando as colunas mais importantes
).orderBy('Keyword', 'Ano' # ordenando por 'Keyword' e depois por 'Ano'
).show()


+-------+----+--------+-----------------------+
|Keyword| Ano|   Likes|Media_Likes_Acumulativa|
+-------+----+--------+-----------------------+
|animals|2009| 1357197|              1357197.0|
|animals|2010|   68133|               712665.0|
|animals|2010|  338601|               587977.0|
|animals|2013|11025176|             3197276.75|
|animals|2014| 1019385|              2761698.4|
|animals|2014| 5743875|     3258727.8333333335|
|animals|2019| 1103713|     2950868.5714285714|
|animals|2020|   74617|            2591337.125|
|animals|2020|    9313|     2304445.5555555555|
|animals|2020|  164337|              2090434.7|
|animals|2020|   94089|     1908948.7272727273|
|animals|2020|   21946|              1751698.5|
|animals|2020|   28863|     1619172.6923076923|
|animals|2020|   73362|      1508757.642857143|
|animals|2020|  282754|     1427024.0666666667|
|animals|2020| 6177588|           1723934.3125|
|animals|2021|   11323|     1623192.4705882352|
|animals|2021|   27102|     1534520.7777